In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from lightgbm import LGBMClassifier

In [ ]:
import json

Train    = pd.read_csv('train.csv')
Test     = pd.read_csv('test.csv')
val_data = pd.read_csv('val.csv')
sub      = pd.read_csv('sample_submission.csv')

with open('label_map.json', 'r') as f:
    map_data = json.load(f)

print("Train shape :", Train.shape)
print("Test shape  :", Test.shape)
print("Val shape   :", val_data.shape)

Train shape : (406709, 56)
Test shape  : (116202, 55)
Val shape   : (58101, 56)


In [ ]:
map_data

{'label2id': {'cover_type_0': 0,
  'cover_type_1': 1,
  'cover_type_2': 2,
  'cover_type_3': 3,
  'cover_type_4': 4,
  'cover_type_5': 5,
  'cover_type_6': 6},
 'id2label': {'0': 'cover_type_0',
  '1': 'cover_type_1',
  '2': 'cover_type_2',
  '3': 'cover_type_3',
  '4': 'cover_type_4',
  '5': 'cover_type_5',
  '6': 'cover_type_6'},
 'class_names': ['cover_type_0',
  'cover_type_1',
  'cover_type_2',
  'cover_type_3',
  'cover_type_4',
  'cover_type_5',
  'cover_type_6']}

In [ ]:
Train

,id,elevation,aspect,slope,horizontal_distance_to_hydrology,vertical_distance_to_hydrology,horizontal_distance_to_roadways,hillshade_9am,hillshade_noon,hillshade_3pm,...,soil_type_id_31,soil_type_id_32,soil_type_id_33,soil_type_id_34,soil_type_id_35,soil_type_id_36,soil_type_id_37,soil_type_id_38,soil_type_id_39,label_id
0,train_0000000,2590.0,56.0,2.0,212.0,-6.0,390.0,220.0,235.0,151.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4
1,train_0000001,2785.0,155.0,18.0,242.0,118.0,3090.0,238.0,238.0,122.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,train_0000002,2595.0,45.0,2.0,153.0,-1.0,391.0,220.0,234.0,150.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4
3,train_0000003,2579.0,132.0,6.0,300.0,-15.0,67.0,230.0,237.0,140.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,train_0000004,2617.0,45.0,9.0,240.0,56.0,666.0,223.0,221.0,133.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
406704,train_0406704,2401.0,157.0,21.0,90.0,15.0,120.0,238.0,238.0,119.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
406705,train_0406705,2396.0,153.0,20.0,85.0,17.0,108.0,240.0,237.0,118.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
406706,train_0406706,2391.0,152.0,19.0,67.0,12.0,95.0,240.0,237.0,119.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
406707,train_0406707,2386.0,159.0,17.0,60.0,7.0,90.0,236.0,241.0,130.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2


In [ ]:
Train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 406709 entries, 0 to 406708
Data columns (total 56 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   id                                  406709 non-null  object 
 1   elevation                           406709 non-null  float64
 2   aspect                              406709 non-null  float64
 3   slope                               406709 non-null  float64
 4   horizontal_distance_to_hydrology    406709 non-null  float64
 5   vertical_distance_to_hydrology      406709 non-null  float64
 6   horizontal_distance_to_roadways     406709 non-null  float64
 7   hillshade_9am                       406709 non-null  float64
 8   hillshade_noon                      406709 non-null  float64
 9   hillshade_3pm                       406709 non-null  float64
 10  horizontal_distance_to_fire_points  406709 non-null  float64
 11  wilderness_area_id_0      

In [ ]:
Test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 116202 entries, 0 to 116201
Data columns (total 55 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   id                                  116202 non-null  object 
 1   elevation                           116202 non-null  float64
 2   aspect                              116202 non-null  float64
 3   slope                               116202 non-null  float64
 4   horizontal_distance_to_hydrology    116202 non-null  float64
 5   vertical_distance_to_hydrology      116202 non-null  float64
 6   horizontal_distance_to_roadways     116202 non-null  float64
 7   hillshade_9am                       116202 non-null  float64
 8   hillshade_noon                      116202 non-null  float64
 9   hillshade_3pm                       116202 non-null  float64
 10  horizontal_distance_to_fire_points  116202 non-null  float64
 11  wilderness_area_id_0      

In [ ]:
Train.describe().T

,count,mean,std,min,25%,50%,75%,max
elevation,406709.0,2959.028413,280.104834,1859.0,2809.0,2995.0,3163.0,3858.0
aspect,406709.0,155.551239,111.921000,0.0,58.0,127.0,260.0,360.0
slope,406709.0,14.096932,7.486499,0.0,9.0,13.0,18.0,66.0
horizontal_distance_to_hydrology,406709.0,269.586466,212.487570,0.0,108.0,218.0,390.0,1397.0
vertical_distance_to_hydrology,406709.0,46.411892,58.334923,-166.0,7.0,29.0,69.0,601.0
horizontal_distance_to_roadways,406709.0,2350.009592,1559.227113,0.0,1104.0,1995.0,3329.0,7117.0
hillshade_9am,406709.0,212.151942,26.774520,0.0,198.0,218.0,231.0,254.0
hillshade_noon,406709.0,223.303642,19.767388,0.0,213.0,226.0,237.0,254.0
hillshade_3pm,406709.0,142.513999,38.231919,0.0,119.0,143.0,168.0,254.0
horizontal_distance_to_fire_points,406709.0,1981.455109,1325.739613,0.0,1024.0,1710.0,2550.0,7173.0


In [ ]:
val_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58101 entries, 0 to 58100
Data columns (total 56 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   id                                  58101 non-null  object 
 1   elevation                           58101 non-null  float64
 2   aspect                              58101 non-null  float64
 3   slope                               58101 non-null  float64
 4   horizontal_distance_to_hydrology    58101 non-null  float64
 5   vertical_distance_to_hydrology      58101 non-null  float64
 6   horizontal_distance_to_roadways     58101 non-null  float64
 7   hillshade_9am                       58101 non-null  float64
 8   hillshade_noon                      58101 non-null  float64
 9   hillshade_3pm                       58101 non-null  float64
 10  horizontal_distance_to_fire_points  58101 non-null  float64
 11  wilderness_area_id_0                58101

In [ ]:
TARGET  = 'label_id'
ID_COL  = 'id'

binary_cols = [col for col in Train.columns
               if col not in [TARGET, ID_COL] and Train[col].nunique() == 2]

continuous_cols = [col for col in Train.select_dtypes(include='number').columns
                   if col not in [TARGET] + binary_cols]

print(f"Continuous cols ({len(continuous_cols)}): {continuous_cols}")
print(f"Binary cols     ({len(binary_cols)}): first 5 → {binary_cols[:5]}")

Continuous cols (10): ['elevation', 'aspect', 'slope', 'horizontal_distance_to_hydrology', 'vertical_distance_to_hydrology', 'horizontal_distance_to_roadways', 'hillshade_9am', 'hillshade_noon', 'hillshade_3pm', 'horizontal_distance_to_fire_points']
Binary cols     (44): first 5 → ['wilderness_area_id_0', 'wilderness_area_id_1', 'wilderness_area_id_2', 'wilderness_area_id_3', 'soil_type_id_0']


In [ ]:
skew_limit = 0.75
pos_skewed = []

for col in continuous_cols:
    skew = Train[col].skew()
    if skew > skew_limit and Train[col].min() >= 0:
        pos_skewed.append(col)
        print(f"  [+] {col}: skew = {skew:.4f}")

for col in pos_skewed:
    Train[col]    = np.log1p(Train[col])
    Test[col]     = np.log1p(Test[col])
    val_data[col] = np.log1p(val_data[col])

  [+] slope: skew = 0.7905
  [+] horizontal_distance_to_hydrology: skew = 1.1388
  [+] horizontal_distance_to_fire_points: skew = 1.2897


In [ ]:
for data in [Train, Test, val_data]:
  data['HV_distance'] = np.sqrt(data['horizontal_distance_to_hydrology']**2 + data['vertical_distance_to_hydrology']**2)
  data['hillshade_diff'] = data['hillshade_3pm'] - data['hillshade_9am']
  data['mean_hillshade'] = data[['hillshade_9am', 'hillshade_noon', 'hillshade_3pm']].mean(axis=1)
  data['mean_Hdistance'] = data[['horizontal_distance_to_hydrology','horizontal_distance_to_roadways','horizontal_distance_to_fire_points']].mean(axis=1)
  data['mean_wilderness'] = data[['wilderness_area_id_0','wilderness_area_id_1','wilderness_area_id_2','wilderness_area_id_3']].mean(axis=1)
  data['dist_elev_v_water'] = data['elevation'] - data['vertical_distance_to_hydrology']
  data['slope_hillshade'] = data['slope'] * data['hillshade_noon']
  data['slope_elevation'] = data['slope'] * data['elevation']

In [ ]:
soil_col = [col for col in Train.columns if 'soil_type' in col.lower()]
for data in [Train, Test, val_data]:
  data['soil_count'] = data[soil_col].sum(axis=1)

In [ ]:
feature_cols = [col for col in Train.columns if col not in [ID_COL, TARGET]]
X_train = Train[feature_cols]
Y_train = Train[TARGET].astype(int)

X_val   = val_data[feature_cols]
Y_val   = val_data[TARGET].astype(int)

X_test  = Test[feature_cols]

In [ ]:
model = RandomForestClassifier(
    n_estimators=480,
    max_depth=None,
    min_samples_split=2,
    max_samples=0.8,
    bootstrap=True,
    n_jobs=-1,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42
)
model.fit(X_train, Y_train)
print("Training complete")

Training complete


In [ ]:
model_2 = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=255,
    feature_fraction=0.8,
    n_jobs=-1,
    random_state=42
)
model_2.fit(X_train,Y_train)

[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.060550 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3986
[LightGBM] [Info] Number of data points in the train set: 406709, number of used features: 60
[LightGBM] [Info] Start training from score -1.010089
[LightGBM] [Info] Start training from score -0.717339
[LightGBM] [Info] Start training from score -2.784235
[LightGBM] [Info] Start training from score -5.349025
[LightGBM] [Info] Start training from score -4.129244
[LightGBM] [Info] Start training from score -3.517541
[LightGBM] [Info] Start training from score -3.339135
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM

LGBMClassifier(feature_fraction=0.8, learning_rate=0.03, n_estimators=2000,
               n_jobs=-1, num_leaves=255, random_state=42)

In [ ]:
Y_pred_val = model.predict(X_val)
acc = accuracy_score(Y_val, Y_pred_val)
print(f"Validation Accuracy: {acc:.4f}  ({acc*100:.2f}%)")
print("\nClassification Report:")
# see actual names, instead of numbers
print(classification_report(Y_val, Y_pred_val, target_names=map_data["class_names"]))

Validation Accuracy: 0.9532  (95.32%)

Classification Report:
              precision    recall  f1-score   support

cover_type_0       0.97      0.94      0.95     21165
cover_type_1       0.95      0.97      0.96     28307
cover_type_2       0.93      0.96      0.94      3508
cover_type_3       0.92      0.85      0.89       288
cover_type_4       0.92      0.77      0.84       952
cover_type_5       0.93      0.88      0.90      1810
cover_type_6       0.97      0.95      0.96      2071

    accuracy                           0.95     58101
   macro avg       0.94      0.90      0.92     58101
weighted avg       0.95      0.95      0.95     58101



In [ ]:
y_pred_val = model_2.predict(X_val)
acc = accuracy_score(Y_val, y_pred_val)
print(f'Validation Accurary: {acc:.4f} ({acc*100:.2f}%)')
print('\nClassification Report:')
print(classification_report(Y_val, y_pred_val, target_names=map_data['class_names']))

[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
Validation Accurary: 0.9725 (97.25%)

Classification Report:
              precision    recall  f1-score   support

cover_type_0       0.98      0.97      0.97     21165
cover_type_1       0.97      0.98      0.98     28307
cover_type_2       0.97      0.97      0.97      3508
cover_type_3       0.94      0.88      0.91       288
cover_type_4       0.93      0.89      0.91       952
cover_type_5       0.95      0.95      0.95      1810
cover_type_6       0.98      0.97      0.97      2071

    accuracy                           0.97     58101
   macro avg       0.96      0.94      0.95     58101
weighted avg       0.97      0.97      0.97     58101



In [ ]:
model1 = model.predict_proba(X_test)

In [ ]:
model2 = model_2.predict_proba(X_test)

[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8


In [ ]:
combined_models = (0.1*model1) + (0.9*model2)

In [ ]:
y_pred_ensemble = np.argmax(combined_models, axis=1)

In [ ]:
sub['label_id'] = np.argmax(model1, axis=1)
sub.to_csv('sample_submission.csv', index=False)